# Model Performance Metrics Beyond NSE

**Companion notebook to:** _Is Nash-Sutcliffe Efficiency Enough? A Python Comparison of Calibration Metrics for Australian Flood Models_

**No direct R source** — written from scratch using formulas from Gupta et al. (2009) and Moriasi et al. (2015).  
Tony Ladson's post [Model performance based on coefficient of efficiency](https://tonyladson.wordpress.com/2019/08/20/model-performance-based-on-coefficient-of-efficiency/) provides the context and performance thresholds.

---

## What this notebook does

Implements four calibration metrics in Python and applies them to synthetic RORB-style hydrographs:

| Metric | What it measures | Blind spot |
|--------|-----------------|------------|
| **NSE** | Overall variance explained | Dominated by high flows; insensitive to volume/timing errors |
| **KGE** | Decomposed error: correlation + bias + variability | More balanced than NSE |
| **PBIAS** | Systematic volume bias (%) | Misses timing errors completely |
| **Peak flow bias** | Accuracy on flood peaks only | Ignores recession and baseflow |

Section 10 is a placeholder for the LYR RORB calibration data.

**Key references:**
- Gupta, H.V. et al. (2009), *Journal of Hydrology* 377(1-2): 80-91
- Moriasi, D.N. et al. (2015), *ASABE* 58(6): 1763-1785
- Kling, H. et al. (2012), *Journal of Hydrology* 468-469: 177-189
- Ladson, A.R. (2019), https://tonyladson.wordpress.com/2019/08/20/model-performance-based-on-coefficient-of-efficiency/


---
## 1. Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.signal import find_peaks

print('numpy:', np.__version__)
print('pandas:', pd.__version__)

numpy: 2.4.6
pandas: 3.0.5


---
## 2. Metric Functions

All four metrics implemented as standalone functions.  
Each takes two 1D arrays: `obs` (observed) and `sim` (simulated).


In [2]:
def nse(obs, sim):
    """
    Nash-Sutcliffe Efficiency (Nash & Sutcliffe 1970).

    NSE = 1 - sum((obs - sim)^2) / sum((obs - mean(obs))^2)

    Range: (-inf, 1]. Perfect = 1. NSE = 0 means model is no better than mean observed.
    """
    obs, sim = np.asarray(obs, float), np.asarray(sim, float)
    numerator = np.sum((obs - sim) ** 2)
    denominator = np.sum((obs - obs.mean()) ** 2)
    if denominator == 0:
        return np.nan
    return 1.0 - numerator / denominator


def kge(obs, sim):
    """
    Kling-Gupta Efficiency (Gupta et al. 2009; modified by Kling et al. 2012).

    KGE = 1 - sqrt((r-1)^2 + (beta-1)^2 + (gamma-1)^2)

    where:
        r     = Pearson correlation coefficient
        beta  = mean(sim) / mean(obs)          [bias ratio]
        gamma = (std(sim)/mean(sim)) / (std(obs)/mean(obs))  [CV ratio, Kling 2012]

    Range: (-inf, 1]. Perfect = 1.
    Returns (KGE, r, beta, gamma) for decomposition.
    """
    obs, sim = np.asarray(obs, float), np.asarray(sim, float)
    r = np.corrcoef(obs, sim)[0, 1]
    beta = sim.mean() / obs.mean()
    gamma = (sim.std() / sim.mean()) / (obs.std() / obs.mean())  # CV ratio
    kge_val = 1.0 - np.sqrt((r - 1)**2 + (beta - 1)**2 + (gamma - 1)**2)
    return kge_val, r, beta, gamma


def pbias(obs, sim):
    """
    Percent Bias (Moriasi et al. 2007).

    PBIAS = 100 * sum(obs - sim) / sum(obs)

    Positive = model underestimates (sim < obs).
    Negative = model overestimates (sim > obs).
    Perfect = 0.
    """
    obs, sim = np.asarray(obs, float), np.asarray(sim, float)
    return 100.0 * np.sum(obs - sim) / np.sum(obs)


def peak_bias(obs, sim, prominence=None):
    """
    Peak flow bias — mean error on the top flood peaks only.

    Uses scipy.signal.find_peaks to identify peaks in observed hydrograph,
    then computes bias at those timesteps.

    peak_bias = mean((sim_peaks - obs_peaks) / obs_peaks) * 100  [%]

    Positive = model overestimates peaks.
    Perfect = 0.
    """
    obs, sim = np.asarray(obs, float), np.asarray(sim, float)
    if prominence is None:
        prominence = obs.max() * 0.1  # default: 10% of max flow
    peak_idx, _ = find_peaks(obs, prominence=prominence)
    if len(peak_idx) == 0:
        return np.nan
    obs_peaks = obs[peak_idx]
    sim_peaks = sim[peak_idx]
    return np.mean((sim_peaks - obs_peaks) / obs_peaks) * 100.0


def score_all(obs, sim, label=''):
    """Compute all four metrics and return a summary dict."""
    kge_val, r, beta, gamma = kge(obs, sim)
    return {
        'Scenario': label,
        'NSE':   round(nse(obs, sim), 3),
        'KGE':   round(kge_val, 3),
        'KGE_r': round(r, 3),
        'KGE_beta': round(beta, 3),
        'KGE_gamma': round(gamma, 3),
        'PBIAS_%': round(pbias(obs, sim), 1),
        'Peak_bias_%': round(peak_bias(obs, sim), 1),
    }


print('Metric functions defined: nse(), kge(), pbias(), peak_bias(), score_all()')

Metric functions defined: nse(), kge(), pbias(), peak_bias(), score_all()


---
## 3. Validation — Unit Tests

Known-answer tests before applying to real data.


In [3]:
# Perfect model: sim == obs  → NSE=1, KGE=1, PBIAS=0, peak_bias=0
obs_test = np.array([1., 2., 5., 3., 2., 1.])
sim_perfect = obs_test.copy()

assert nse(obs_test, sim_perfect) == 1.0,          'NSE perfect test failed'
assert np.isclose(kge(obs_test, sim_perfect)[0], 1.0),  'KGE perfect test failed'
assert pbias(obs_test, sim_perfect) == 0.0,          'PBIAS perfect test failed'

# Mean model: sim == mean(obs) → NSE=0
sim_mean = np.full_like(obs_test, obs_test.mean())
assert abs(nse(obs_test, sim_mean)) < 1e-10,         'NSE mean model test failed'

# Systematic overestimate: sim = 2*obs → PBIAS = -100%
sim_over = obs_test * 2
assert pbias(obs_test, sim_over) == -100.0,          'PBIAS overestimate test failed'

print('All unit tests passed.')

All unit tests passed.


---
## 4. Synthetic RORB-Style Hydrographs

Four scenarios, each with a different known deficiency — matching common RORB calibration failure modes:

| Scenario | Deficiency | What it looks like |
|----------|-----------|--------------------|
| A | **Perfect** | Baseline — sim matches obs exactly |
| B | **Volume error** | Sim matches shape/timing, but volumes are ~30% too high |
| C | **Timing error** | Sim peak arrives 6 hours early |
| D | **Peak bias** | Sim underestimates the flood peak by 40% |


In [4]:
# Synthetic observed hydrograph — single-peaked, 72-hour event, 1-hour timestep
t = np.arange(0, 73)  # hours

def synthetic_hydrograph(t, peak_flow=100., peak_time=24., rise_k=0.3, fall_k=0.12, baseflow=2.):
    """Asymmetric hydrograph: rapid rise, exponential recession."""
    rise = np.where(t <= peak_time,
                    peak_flow * np.exp(-rise_k * (peak_time - t)),
                    0.)
    fall = np.where(t > peak_time,
                    peak_flow * np.exp(-fall_k * (t - peak_time)),
                    0.)
    return np.maximum(rise + fall, baseflow)

obs = synthetic_hydrograph(t, peak_flow=100., peak_time=24.)

# Scenario A — perfect
sim_A = obs.copy()

# Scenario B — volume error (+30% systematic overestimate)
sim_B = obs * 1.30

# Scenario C — 6-hour timing advance (peak arrives early)
sim_C = synthetic_hydrograph(t, peak_flow=100., peak_time=18.)  # 6hr earlier

# Scenario D — peak underestimate (40% lower peak, same timing)
sim_D = synthetic_hydrograph(t, peak_flow=60., peak_time=24.)

scenarios = {
    'A — Perfect':       sim_A,
    'B — Volume error':  sim_B,
    'C — Timing error':  sim_C,
    'D — Peak underest': sim_D,
}

print(f'Observed peak flow: {obs.max():.0f} m³/s at t={t[np.argmax(obs)]} hr')
for name, sim in scenarios.items():
    print(f'  {name}: peak = {sim.max():.0f} m³/s at t={t[np.argmax(sim)]} hr')

Observed peak flow: 100 m³/s at t=24 hr
  A — Perfect: peak = 100 m³/s at t=24 hr
  B — Volume error: peak = 130 m³/s at t=24 hr
  C — Timing error: peak = 100 m³/s at t=18 hr
  D — Peak underest: peak = 60 m³/s at t=24 hr


---
## 5. Plot the Four Scenarios

In [5]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharey=True)

for ax, (name, sim) in zip(axes.flat, scenarios.items()):
    ax.plot(t, obs, 'k-', lw=2, label='Observed')
    ax.plot(t, sim, 'steelblue', lw=1.5, linestyle='--', label='Simulated')
    ax.fill_between(t, obs, sim, alpha=0.15, color='red')
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.set_xlabel('Time (hours)', fontsize=9)
    ax.set_ylabel('Flow (m³/s)', fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Synthetic RORB-style hydrographs — four calibration scenarios', fontsize=12)
plt.tight_layout()
plt.savefig('../../images/2026-04_metrics-synthetic-hydrographs.png', dpi=150, bbox_inches='tight')
plt.show()

<Figure size ... with Axes>

---
## 6. Score All Scenarios

In [6]:
results = [score_all(obs, sim, label=name) for name, sim in scenarios.items()]
results_df = pd.DataFrame(results)

# Display with colour highlighting
print(results_df.to_string(index=False))

         Scenario   NSE   KGE  KGE_r  KGE_beta  KGE_gamma  PBIAS_%  Peak_bias_%
      A — Perfect 1.000 1.000  1.000      1.00      1.000      0.0          0.0
 B — Volume error 0.865 0.700  1.000      1.30      1.000    -30.0         30.0
 C — Timing error 0.091 0.545  0.545      1.00      1.000      0.0        -51.3
D — Peak underest 0.760 0.617  1.000      0.62      0.953     38.0        -40.0


---
## 7. Which Metric Catches Which Problem?

Visualise the metric scores as a heatmap — shows at a glance which metric flags each deficiency.


In [7]:
# Subset to the four headline metrics for the heatmap
metrics = ['NSE', 'KGE', 'PBIAS_%', 'Peak_bias_%']
heatmap_df = results_df.set_index('Scenario')[metrics]

fig, ax = plt.subplots(figsize=(8, 3.5))

# Colour each column on its OWN scale, since NSE/KGE (bounded ~[0,1], higher
# = better) and PBIAS/Peak_bias (unbounded %, |value| near 0 = better) are
# not comparable in raw magnitude. A shared linear colour scale across all
# four columns makes a poor NSE (e.g. 0.1) read as "green" just because 0.1
# is numerically small next to a 30% bias -- misleading. Instead, normalise
# each column independently to a 0 (best-in-column) to 1 (worst-in-column)
# "badness" score before colouring, so the map shows relative discrimination
# within each metric, not absolute magnitude across different units.
display_df = heatmap_df.copy()
display_df['PBIAS_%'] = display_df['PBIAS_%'].abs()
display_df['Peak_bias_%'] = display_df['Peak_bias_%'].abs()

badness = pd.DataFrame(index=display_df.index, columns=display_df.columns, dtype=float)
for col in ['NSE', 'KGE']:  # higher raw value = better
    lo, hi = display_df[col].min(), display_df[col].max()
    badness[col] = 0.0 if hi == lo else (hi - display_df[col]) / (hi - lo)
for col in ['PBIAS_%', 'Peak_bias_%']:  # lower |value| = better
    lo, hi = display_df[col].min(), display_df[col].max()
    badness[col] = 0.0 if hi == lo else (display_df[col] - lo) / (hi - lo)

im = ax.imshow(badness.values.astype(float), cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=1)

ax.set_xticks(range(len(metrics)))
ax.set_xticklabels(metrics, fontsize=10)
ax.set_yticks(range(len(heatmap_df)))
ax.set_yticklabels(heatmap_df.index, fontsize=9)

# Annotate cells with the actual metric value (not the normalised colour score)
for i in range(len(heatmap_df)):
    for j, m in enumerate(metrics):
        val = heatmap_df.iloc[i, j]
        ax.text(j, i, f'{val:.1f}', ha='center', va='center', fontsize=9)

plt.colorbar(im, ax=ax, label='Relative performance within column\n(green = best-in-column, red = worst-in-column)')
ax.set_title('Metric scores by scenario — which metric catches which deficiency?', fontsize=11)
plt.tight_layout()
plt.savefig('../../images/2026-04_metrics-heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

<Figure size ... with Axes>

---
## 8. KGE Decomposition — What's Driving the Score?

KGE breaks into three components: correlation (r), bias ratio (β), and CV ratio (γ).  
This shows *why* KGE is low, not just that it is.


In [8]:
kge_decomp = results_df[['Scenario', 'KGE', 'KGE_r', 'KGE_beta', 'KGE_gamma']].copy()
kge_decomp.columns = ['Scenario', 'KGE', 'r (correlation)', 'β (bias ratio)', 'γ (CV ratio)']
print(kge_decomp.to_string(index=False))

# TODO: add bar chart decomposition

         Scenario   KGE  r (correlation)  β (bias ratio)  γ (CV ratio)
      A — Perfect 1.000            1.000            1.00         1.000
 B — Volume error 0.700            1.000            1.30         1.000
 C — Timing error 0.545            0.545            1.00         1.000
D — Peak underest 0.617            1.000            0.62         0.953


---
## 9. Performance Thresholds for Reporting

Reference thresholds from Moriasi et al. (2015) — used in ARR 2019 calibration reporting:

| Metric | Very Good | Good | Satisfactory | Unsatisfactory |
|--------|-----------|------|--------------|----------------|
| NSE | > 0.80 | 0.70–0.80 | 0.50–0.70 | < 0.50 |
| PBIAS (stream) | < ±5% | ±5–±10% | ±10–±15% | > ±15% |
| KGE | > 0.75 | 0.65–0.75 | 0.50–0.65 | < 0.50 |

Note: Tony Ladson's post documents how different agencies (GBR, North Sea) use different thresholds — the choice of threshold matters as much as the choice of metric.


In [9]:
def classify_nse(val):
    if val > 0.80: return 'Very Good'
    if val > 0.70: return 'Good'
    if val > 0.50: return 'Satisfactory'
    return 'Unsatisfactory'

def classify_pbias(val):
    v = abs(val)
    if v < 5:  return 'Very Good'
    if v < 10: return 'Good'
    if v < 15: return 'Satisfactory'
    return 'Unsatisfactory'

def classify_kge(val):
    if val > 0.75: return 'Very Good'
    if val > 0.65: return 'Good'
    if val > 0.50: return 'Satisfactory'
    return 'Unsatisfactory'

classification_df = results_df[['Scenario', 'NSE', 'KGE', 'PBIAS_%']].copy()
classification_df['NSE_class'] = classification_df['NSE'].apply(classify_nse)
classification_df['KGE_class'] = classification_df['KGE'].apply(classify_kge)
classification_df['PBIAS_class'] = classification_df['PBIAS_%'].apply(classify_pbias)

print(classification_df.to_string(index=False))

         Scenario   NSE   KGE  PBIAS_%      NSE_class    KGE_class    PBIAS_class
      A — Perfect 1.000 1.000      0.0      Very Good    Very Good      Very Good
 B — Volume error 0.865 0.700    -30.0      Very Good         Good Unsatisfactory
 C — Timing error 0.091 0.545      0.0 Unsatisfactory Satisfactory      Very Good
D — Peak underest 0.760 0.617     38.0           Good Satisfactory Unsatisfactory


---
## 10. TODO — LYR RORB Calibration Application

This section is a placeholder for the Lower Yarra River (LYR) RORB calibration analysis.  
Lindsay to populate from the bespoke post-processing code in the project report.


In [ ]:
# TODO: LYR RORB Calibration Application
# ----------------------------------------
# Expected inputs from the project post-processing code:
#   - observed hydrograph: datetime index, flow in m³/s
#   - simulated hydrograph(s): same format, one column per calibration event
#
# Suggested workflow:
#
#   1. Load paired obs/sim from the LYR project CSVs
#      obs_lyr = pd.read_csv('data/lyr_observed.csv', parse_dates=['datetime'], index_col='datetime')
#      sim_lyr = pd.read_csv('data/lyr_simulated.csv', parse_dates=['datetime'], index_col='datetime')
#
#   2. Align on common timestep
#      merged = obs_lyr.join(sim_lyr, how='inner')
#
#   3. Score each calibration event
#      for event in events:
#          print(score_all(merged.loc[event, 'obs'], merged.loc[event, 'sim'], label=event))
#
#   4. Compare across calibration and validation periods
#
#   5. Plot observed vs simulated with metric annotations

print('LYR RORB calibration application — to be completed')

---
## 11. Helper — Load RORB Output CSV

Stub function for loading RORB hydrograph output once the LYR data is available.


In [ ]:
def load_rorb_output(filepath, obs_col='observed_m3s', sim_col='modelled_m3s',
                     datetime_col='datetime'):
    """
    Load a paired observed/simulated RORB output CSV.

    Expected CSV format:
        datetime,observed_m3s,modelled_m3s
        2011-01-10 00:00,0.0,0.0
        2011-01-10 01:00,12.3,14.1
        ...

    Returns
    -------
    obs : np.ndarray
    sim : np.ndarray
    dt  : pd.DatetimeIndex
    """
    df = pd.read_csv(filepath, parse_dates=[datetime_col])
    df = df.dropna(subset=[obs_col, sim_col])
    return (
        df[obs_col].values,
        df[sim_col].values,
        df[datetime_col],
    )


# Usage (once LYR data is available):
# obs_lyr, sim_lyr, dt_lyr = load_rorb_output('data/urbs_calibration_event_2011jan.csv')
# print(score_all(obs_lyr, sim_lyr, label='LYR 2011 Jan event'))

print('load_rorb_output() defined and ready.')

---
## References

- Gupta, H.V., Kling, H., Yilmaz, K.K. and Martinez, G.F. (2009). Decomposition of the mean squared error and NSE: Implications for improving hydrological modelling. *Journal of Hydrology* 377(1–2): 80–91.
- Kling, H., Fuchs, M. and Paulin, M. (2012). Runoff conditions in the upper Danube basin under an ensemble of climate change scenarios. *Journal of Hydrology* 468–469: 177–189.
- Moriasi, D.N. et al. (2015). Hydrologic and Water Quality Models: Performance Measures and Evaluation Criteria. *Transactions of the ASABE* 58(6): 1763–1785.
- Nash, J.E. and Sutcliffe, J.V. (1970). River flow forecasting through conceptual models. *Journal of Hydrology* 10(3): 282–290.
- Ladson, A.R. (2019). Model performance based on coefficient of efficiency. https://tonyladson.wordpress.com/2019/08/20/model-performance-based-on-coefficient-of-efficiency/
